# SCDMN-comma2k19 — Colab runner

Runs the full pipeline on a Colab T4/L4 GPU:

1. Clone repo, mount Drive (persistent data + checkpoints).
2. Install deps.
3. Download + prepare comma2k19 chunk(s) **once** (skipped on rerun).
4. Train Single baseline.
5. Train SCDMN-Sliced.
6. Print comparison table.

Drive layout:
```
/content/drive/MyDrive/SCDMN-comma2k19/
  data_cache/     # chunks + frames/ + frames.csv (persistent)
  runs/           # checkpoints + logs (persistent)
```

## 1. Clone repo and mount Drive

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

REPO_DIR = '/content/SCDMN-comma2k19'
REPO_URL = 'https://github.com/YOUR_USER/SCDMN-comma2k19.git'  # TODO: set
if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone', REPO_URL, REPO_DIR])
else:
    subprocess.check_call(['git', '-C', REPO_DIR, 'pull'])
%cd $REPO_DIR

In [ ]:
PERSIST = '/content/drive/MyDrive/SCDMN-comma2k19'
DATA_ROOT = f'{PERSIST}/data_cache'
RUNS_DIR = f'{PERSIST}/runs'
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
print('data:', DATA_ROOT)
print('runs:', RUNS_DIR)

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq install -y aria2 unzip > /dev/null
!nvidia-smi | head -20

## 3. Download + prepare data (one-time)

Checks for `frames.csv` under Drive first — if present, skips the ~10 GB download and HEVC decode.

In [ ]:
import os
CSV = f'{DATA_ROOT}/frames.csv'
if not os.path.exists(CSV):
    !python -m scripts.download_comma2k19 --data_root $DATA_ROOT --chunks 1
    !python -m scripts.prepare_frames    --data_root $DATA_ROOT --fps 10 --image_size 224
    !python -m scripts.label_context     --csv $CSV
else:
    print(f'frames.csv exists at {CSV} — skipping download/prepare/label')
    !python -m scripts.label_context --csv $CSV

## 4. Train Single baseline

In [ ]:
!python -m experiments.run_experiment \
    --model single \
    --csv_path $CSV \
    --save_dir $RUNS_DIR \
    --drive_save_dir $RUNS_DIR \
    --epochs 30 --batch_size 64 --lr 0.05

## 5. Train SCDMN-Sliced

In [ ]:
!python -m experiments.run_experiment \
    --model sliced \
    --csv_path $CSV \
    --save_dir $RUNS_DIR \
    --drive_save_dir $RUNS_DIR \
    --epochs 30 --batch_size 64 --lr 0.05 \
    --sparsity 0.5 --freeze_epoch 5

## 6. Compare per-context MAE

In [ ]:
!python -m utils.analyze \
    --single $RUNS_DIR/single_summary.json \
    --sliced $RUNS_DIR/sliced_s0.5_f5_summary.json \
    --out    $RUNS_DIR/comma2k19_summary.md